In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

# 벡터 DB : Chroma vs Pinecone
- Chroma : 인메모리 vector DB, 로컬메모리 vector DB
- Pinecone : 클라우드 vector DB (Pinecone console에 api key 생성 -> .env (PINECONE_API_KEY등록)

## 0. 패키지 설치

In [2]:
%pip install -q pinecone-client langchain-pinecone

Note: you may need to restart the kernel to use updated packages.


## 1. Knowledge Base 구성을 위한 데이터 생성

In [3]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader('./tax_docs/with_markdown.docx')
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200
)
document_list = loader.load_and_split(text_splitter=text_splitter)

In [4]:
len(document_list)

225

In [5]:
# embedding : upstage embedding-query
# https://python.langchain.com/v0.2/docs/integrations/text_embedding/upstage
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv()
embedding = UpstageEmbeddings(
    model="solar-embedding-1-large"
    # model="embedding-query"
)

In [10]:
%%time
# pinecone vector database
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

pc = Pinecone()
# 데이터를 처음 업로드할 때
index_name = "tax-index-markdown"
database = PineconeVectorStore.from_documents(
    documents=document_list,
    embedding=embedding,
    index_name=index_name
)
# 업로드한 벡터DB 가져올 때
# database = PineconeVectorStore(
#     embedding=embedding, # 질문을 임베딩하여 유사도 검색
#     index_name=index_name
# )

PineconeApiException: (400)
Reason: Bad Request
HTTP response headers: HTTPHeaderDict({'Date': 'Wed, 30 Jul 2025 07:48:37 GMT', 'Content-Type': 'application/json', 'Content-Length': '104', 'Connection': 'keep-alive', 'x-pinecone-request-latency-ms': '1484', 'x-pinecone-request-id': '2420572842556135550', 'x-envoy-upstream-service-time': '30', 'server': 'envoy'})
HTTP response body: {"code":3,"message":"Vector dimension 4096 does not match the dimension of the index 1024","details":[]}


In [8]:
pc.delete_index("tax-index-table")

In [9]:
from pinecone import ServerlessSpec
pc.create_index(
    name="tax-index-table",
    dimension=4096,                # UpstageEmbeddings 모델 차원
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1")
)

{
    "name": "tax-index-table",
    "metric": "cosine",
    "host": "tax-index-table-1rmfod6.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 4096,
    "deletion_protection": "disabled",
    "tags": null
}

In [11]:
from pinecone import Pinecone

pc = Pinecone(api_key="pcsk_3NRp9v_RNS4vwStuvDqi2X7TD19NWvkjuDQE9JPB5xy8FzBb3MHEPaZXUhNr11uyQaXZ8u")
desc = pc.describe_index("tax-index-table")
print("Current index dimension:", desc.dimension)

Current index dimension: 4096


In [12]:
from pinecone import ServerlessSpec

pc.delete_index("tax-index-table")  # 기존 인덱스 삭제
pc.create_index(
    name="tax-index-table",
    dimension=4096,
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1")
)

{
    "name": "tax-index-table",
    "metric": "cosine",
    "host": "tax-index-table-1rmfod6.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 4096,
    "deletion_protection": "disabled",
    "tags": null
}

In [13]:
%%time
# pinecone vector database
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

pc = Pinecone()
# 데이터를 처음 업로드할 때
index_name = "tax-index-markdown"
database = PineconeVectorStore.from_documents(
    documents=document_list,
    embedding=embedding,
    index_name=index_name
)
# 업로드한 벡터DB 가져올 때
# database = PineconeVectorStore(
#     embedding=embedding, # 질문을 임베딩하여 유사도 검색
#     index_name=index_name
# )

PineconeApiException: (400)
Reason: Bad Request
HTTP response headers: HTTPHeaderDict({'Date': 'Wed, 30 Jul 2025 07:51:51 GMT', 'Content-Type': 'application/json', 'Content-Length': '104', 'Connection': 'keep-alive', 'x-pinecone-request-latency-ms': '1583', 'x-pinecone-request-id': '3932918892783500116', 'x-envoy-upstream-service-time': '29', 'server': 'envoy'})
HTTP response body: {"code":3,"message":"Vector dimension 4096 does not match the dimension of the index 1024","details":[]}
